# Analysis – Spatial Representations
**Michon Linde et al., Nature Communications**  
*"The Intermediate Hippocampus Integrates Shock-Observation and Spatial Information during Observational Fear Memory"*

Covers: **Fig. 4b–c** · **Extended Data Fig. 4a–b**

> Set `Folder_path` below to the directory containing the summary data tables.

## Imports

In [ ]:
import os
import numpy as np
import pandas as pd

import statsmodels.api as sm
import pingouin as pg

import seaborn as sns
from matplotlib import pyplot as plt

## Data loading

In [ ]:
# ── Set this path to the folder containing the summary data tables ──────────
Folder_path = "/data07/Fred/Ctx_Hpc/summaries/NatCom/"

# Per-neuron spatial firing properties (peak rate, spatial info) across phases
df = pd.read_parquet(os.path.join(Folder_path, 'table_spatial_firing.parquet'))

# Per-neuron pairwise correlation distances between all phase/context combinations
df_mds = pd.read_parquet(os.path.join(Folder_path, 'table_spatial_similarities.parquet'))

# Per-neuron baseline-to-recall correlation distance, separately for safe and shock context
df_mds_ctx = pd.read_parquet(os.path.join(Folder_path, 'table_spatial_PrePostsimilarities.parquet'))

# Per-neuron safe–shock context correlation distance, separately for baseline and recall
df_mds_ph = pd.read_parquet(os.path.join(Folder_path, 'table_spatial__context_similarities.parquet'))

print("Loaded spatial representation tables:")
print(f"  df         : {df.shape[0]} rows — spatial firing properties")
print(f"  df_mds     : {df_mds.shape[0]} rows — pairwise correlation distances")
print(f"  df_mds_ctx : {df_mds_ctx.shape[0]} rows — baseline-to-recall correlation distance")
print(f"  df_mds_ph  : {df_mds_ph.shape[0]} rows — safe–shock context correlation distance")
print()
print("Pyramidal neurons per subregion:")
print(df.query("neuron_type == 'pyr'").groupby('pole').size().to_string())

---
## Figure 4b — All animals
**Baseline-to-recall spatial map correlation distance, safe vs. shock context.**  
1 − *r* between spatial rate maps estimated during solo baseline and recall exploration,  
separately for the safe (green) and shock (purple) context.  
Lower values indicate greater map stability across learning.  
Left panels show all animals pooled.

In [ ]:
tmp_mds = df_mds.groupby(['rat', 'cluster']).first()

for pole in ['dorsal', 'intermediate', 'ventral']:
    fig, ax = plt.subplots(1, 1, figsize=(1, 3))

    # Paired lines (one per animal)
    for l in np.array(tmp_mds.query("pole == '{}'".format(pole))[
                          ['ctrl-context_ctrl-test', 'shock-context_shock-test']]):
        ax.plot(l, color='gray', alpha=0.1)

    sns.boxplot(x='context', y='Changes(baseline-recall)',
                order=['ctrl', 'shock'],
                palette=['darkgreen', 'rebeccapurple'],
                boxprops=dict(alpha=0.4), fliersize=0.0,
                data=df_mds_ctx.query("pole == '{}'".format(pole)), ax=ax)
    sns.stripplot(x='context', y='Changes(baseline-recall)',
                  order=['ctrl', 'shock'],
                  palette=['darkgreen', 'rebeccapurple'],
                  size=8, alpha=0.3, dodge=False, marker='^',
                  data=df_mds_ctx.query("pole == '{}'".format(pole)), ax=ax)

    ax.set(ylabel='correlation distance (baseline–recall)',
           xlabel='', xticklabels=['safe', 'shock'],
           ylim=(0, 1.85),
           title='{} hippocampus'.format(pole))
    sns.despine(offset=True, trim=True)

## Figure 4b — RECALLER / NON-RECALLER split
Same analysis as above, further divided by recall status (right panels of Fig. 4b).

In [ ]:
for pole in ['dorsal', 'intermediate', 'ventral']:
    fig, ax = plt.subplots(1, 1, figsize=(2, 3))

    sns.boxplot(x='context', y='Changes(baseline-recall)',
                order=['ctrl', 'shock'],
                hue='context_learning', hue_order=['not learned', 'learned'],
                palette=['grey', 'orchid'],
                boxprops=dict(alpha=0.4), fliersize=0.0,
                data=df_mds_ctx.query("pole == '{}'".format(pole)), ax=ax)
    sns.stripplot(x='context', y='Changes(baseline-recall)',
                  order=['ctrl', 'shock'],
                  hue='context_learning', hue_order=['not learned', 'learned'],
                  palette=['grey', 'orchid'],
                  size=8, alpha=0.3, dodge=True, marker='^',
                  data=df_mds_ctx.query("pole == '{}'".format(pole)), ax=ax)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], ['NON-RECALLER', 'RECALLER'], frameon=False)
    ax.set(ylabel='correlation distance (baseline–recall)',
           yticks=[], xlabel='',
           xticklabels=['safe', 'shock'],
           ylim=(0, 1.85),
           title='{} hippocampus'.format(pole))
    sns.despine(offset=True, trim=True)

---
## Figure 4c
**Greater safe–shock context separation at recall in RECALLER animals (intermediate hippocampus).**  
Safe–shock correlation distance (1 − *r*) between spatial rate maps of the two contexts,  
computed separately during solo baseline and recall, split by recall status.  
Higher values indicate more distinct representations of the safe vs. shock context.

In [ ]:
for pole in ['dorsal', 'intermediate', 'ventral']:
    fig, ax = plt.subplots(1, 1, figsize=(2, 3))

    sns.boxplot(x='phase', y='context_dissimilarity',
                order=['context', 'test'],
                hue='context_learning', hue_order=['not learned', 'learned'],
                palette=['gray', 'orchid'],
                boxprops=dict(alpha=0.4), fliersize=0.0,
                data=df_mds_ph.query("pole == '{}'".format(pole)), ax=ax)
    sns.stripplot(x='phase', y='context_dissimilarity',
                  order=['context', 'test'],
                  hue='context_learning', hue_order=['not learned', 'learned'],
                  palette=['gray', 'orchid'],
                  size=8, alpha=0.3, dodge=True, marker='^',
                  data=df_mds_ph.query("pole == '{}'".format(pole)), ax=ax)

    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles[:2], ['NON-RECALLER', 'RECALLER'], frameon=False)
    ax.set(ylabel='safe–shock correlation distance',
           yticks=[], xlabel='',
           xticklabels=['baseline', 'recall'],
           ylim=(0, 1.85),
           title='{} hippocampus'.format(pole))
    sns.despine(offset=True, trim=True)

### Statistics — Fig. 4b–c | Linear mixed models: correlation distance ~ context × recall status

In [ ]:
stats = []
stats.append("=== Fig. 4b–c — Linear mixed models: spatial map correlation distances ===")

for pole in ['dorsal', 'intermediate', 'ventral']:
    stats.append("")
    stats.append("━━━ {} hippocampus ━━━".format(pole))

    # ── Fig. 4b: baseline-to-recall distance ~ context × recall status ────────
    tmp = (df_mds_ctx.query("pole == '{}'".format(pole))
           [['rat', 'cluster', 'context_learning', 'context', 'Changes(baseline-recall)', 'response']]
           .dropna()
           .rename(columns={'Changes(baseline-recall)': 'changes'}))
    tmp['dummy_context']          = ['safe' if s == 'ctrl' else 'shock' for s in tmp['context']]
    tmp['dummy_context_learning'] = ['RECALLER' if s == 'learned' else 'NON-RECALLER'
                                      for s in tmp['context_learning']]
    tmp['rat']     = tmp.rat.astype('str')
    tmp['cluster'] = tmp.cluster.astype('str')
    tmp['response'] = (tmp['response'].astype('category')
                        .cat.reorder_categories(['rest', 'ShockObs+']))

    model   = sm.MixedLM.from_formula(
        'changes ~ C(dummy_context) * C(dummy_context_learning)',
        data=tmp, groups=np.ones(tmp.shape[0]),
        vc_formula={'rat': 'C(rat)'})
    results = model.fit()
    summ    = results.summary()
    stats.append("Fig. 4b — baseline-to-recall distance ~ context × recall status:")
    stats.append(summ.tables[0].to_string())
    stats.append(summ.tables[1].to_string())

    # ── Fig. 4b: shock-responsive vs. unresponsive neurons ────────────────────
    model2   = sm.MixedLM.from_formula(
        'changes ~ C(dummy_context) * C(dummy_context_learning) * C(response)',
        data=tmp, groups=np.ones(tmp.shape[0]),
        vc_formula={'rat': 'C(rat)'})
    results2 = model2.fit()
    summ2    = results2.summary()
    stats.append("Fig. 4b — effect of ShockObs+ response on baseline-to-recall distance:")
    stats.append(summ2.tables[1].to_string())

    # ── Fig. 4c: safe–shock distance ~ phase × recall status ─────────────────
    tmp_ph = (df_mds_ph.query("pole == '{}'".format(pole))
              [['rat', 'cluster', 'context_learning', 'phase', 'context_dissimilarity']]
              .dropna())
    tmp_ph['dummy_phase']   = ['baseline' if s == 'context' else 'recall' for s in tmp_ph['phase']]
    tmp_ph['dummy_recall']  = ['RECALLER' if s == 'learned' else 'NON-RECALLER'
                                for s in tmp_ph['context_learning']]
    tmp_ph['rat']     = tmp_ph.rat.astype('str')
    tmp_ph['cluster'] = tmp_ph.cluster.astype('str')

    model3   = sm.MixedLM.from_formula(
        'context_dissimilarity ~ C(dummy_phase) * C(dummy_recall)',
        data=tmp_ph, groups=np.ones(tmp_ph.shape[0]),
        vc_formula={'rat': 'C(rat)'})
    results3 = model3.fit()
    summ3    = results3.summary()
    stats.append("Fig. 4c — safe–shock distance ~ phase × recall status:")
    stats.append(summ3.tables[1].to_string())

for line in stats:
    print(line)

---
## Extended Data Figure 4a
**Spatial tuning properties of shock-observation-excited vs. unresponsive pyramidal neurons (solo baseline).**  
Peak field rate and spatial information during solo baseline exploration of the shock context,  
split by subregion and by whether neurons were significantly excited by shock observation (ShockObs⁺)  
or not (unresponsive/inhibited).

In [ ]:
var_list  = ['field_peak_rate', 'spatial_info']
labels    = ['peak field rate (spikes/s)', 'spatial information (bits/spike)']
ylims     = [(0, 35), (0, 3.5)]
x_var     = 'response'
order     = ['ShockObs+', 'rest']
xtick_lab = ['ShockObs⁺', 'unresponsive']

tmp = df.query("neuron_type == 'pyr' and field_mean_rate >= 0.5")

for var, ylabel, ylim in zip(var_list, labels, ylims):
    for po in ['dorsal', 'intermediate', 'ventral']:
        fig, ax = plt.subplots(1, 1, figsize=(1.0, 2.0))
        sns.boxplot(x=x_var, y=var, order=order,
                    palette=['coral', 'lightgrey'],
                    boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax,
                    data=tmp.query("phase == 'context' and pole == '{}' and condition == 'shock'".format(po)))
        sns.stripplot(x=x_var, y=var, order=order,
                      palette=['coral', 'lightgrey'],
                      size=6, alpha=0.5, dodge=False, ax=ax,
                      data=tmp.query("phase == 'context' and pole == '{}' and condition == 'shock'".format(po)))
        ax.set(title='{}'.format(po), ylabel=ylabel,
               xlabel='', xticklabels=xtick_lab, ylim=ylim)
        plt.xticks(rotation=25, ha='right')
        sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 4a | Independent t-tests: spatial tuning (ShockObs⁺ vs. unresponsive)

In [ ]:
stats = []
tmp = df.query("neuron_type == 'pyr'")

for var, label in [('field_peak_rate', 'peak field rate'),
                    ('spatial_info',    'spatial information')]:
    stats.append("")
    stats.append("=== Ext. Data Fig. 4a — {} (ShockObs+ vs. unresponsive) ===".format(label))
    for pole in ['dorsal', 'intermediate', 'ventral']:
        results = pg.ttest(
            tmp.query("phase == 'context' and condition == 'shock' and response == 'ShockObs+' and pole == '{}'".format(pole))[var],
            tmp.query("phase == 'context' and condition == 'shock' and response == 'rest'    and pole == '{}'".format(pole))[var],
            paired=False)
        stats.append("  {} — {}".format(pole, label))
        stats.append(results.to_string())
        stats.append("")

for line in stats:
    print(line)

---
## Extended Data Figure 4b
**No systematic change in peak field rate or spatial information across learning.**  
Peak field rate (top) and spatial information (bottom) for spatial rate maps estimated during  
solo baseline and recall, separately for the safe and shock context and each hippocampal subregion.

In [ ]:
plt.rc('xtick', labelsize=12)

tmp   = df.query("neuron_type == 'pyr' and field_mean_rate >= 0.5")
order = ['context', 'test']

for var, ylabel, ylim in [('field_peak_rate', 'peak field rate (spikes/s)', (0, 40)),
                           ('spatial_info',    'spatial information (bits/spike)', (0, 3.75))]:

    fig, ax = plt.subplots(1, 3, figsize=(6, 3), sharex=True, sharey=True)

    for p, po in enumerate(['dorsal', 'intermediate', 'ventral']):
        data_po = tmp.query("(phase == 'context' or phase == 'test') and pole == '{}'".format(po))
        sns.boxplot(x='phase', y=var, hue='condition',
                    order=order, hue_order=['ctrl', 'shock'],
                    palette=['darkgreen', 'rebeccapurple'],
                    boxprops=dict(alpha=0.4), fliersize=0.0, ax=ax[p],
                    data=data_po)
        sns.stripplot(x='phase', y=var, hue='condition',
                      order=order, hue_order=['ctrl', 'shock'],
                      palette=['darkgreen', 'rebeccapurple'],
                      size=6, alpha=0.5, dodge=True, ax=ax[p],
                      data=data_po)
        ax[p].set(title='{} hippocampus'.format(po),
                  xlabel='', ylabel='',
                  xticklabels=['baseline', 'recall'],
                  ylim=ylim)
        ax[p].get_legend().remove()

    ax[0].set(ylabel=ylabel)
    sns.despine(offset=True, trim=True)

### Statistics — Extended Data Fig. 4b | Two-way ANOVA: peak rate and spatial information ~ context × phase

In [ ]:
stats = []
tmp = df.query("neuron_type == 'pyr'")

for pole in ['dorsal', 'intermediate', 'ventral']:
    stats.append("")
    stats.append("━━━ {} hippocampus ━━━".format(pole))

    for var, label in [('field_peak_rate', 'peak field rate'),
                        ('spatial_info',    'spatial information')]:
        data = tmp.query("(phase == 'context' or phase == 'test') and pole == '{}'".format(pole))[
            [var, 'condition', 'phase']]

        results  = pg.anova(data=data, dv=var, between=['condition', 'phase'], ss_type=2)
        posthoc1 = pg.pairwise_tests(dv=var, between=['condition', 'phase'],
                                      padjust='holm', effsize='cohen', data=data)
        stats.append("  {} — two-way ANOVA: context × phase".format(label))
        stats.append(results.to_string())
        stats.append("")
        stats.append(posthoc1.to_string())
        stats.append("")

for line in stats:
    print(line)